# Part 3 — Ranking & Filtering

| Method | Status |
|---|---|
| 1 | TF-IDF + cosine similarity
| 2 | BM25 |
| 1c | to do |

Rankings are registered in `RANKINGS_LIST` (`myapp/search/algorithms.py`) and selected with `search_in_corpus(query, index, ranking=...)`.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "myapp").exists())
sys.path.insert(0, str(ROOT))

## 1. Load the index and the corpus

In [2]:
import pandas as pd
from myapp.search.indexing import load_index
from myapp.search.algorithms import process_query, conjunctive_retrieve, search_in_corpus, RANKINGS_LIST
from myapp.search.evaluation import (labels_in_rank_order, count_relevant, precision_at_k,
                                     average_precision, average_precision_at_k, mean_average_precision,
                                     reciprocal_rank, mean_reciprocal_rank, ndcg_at_k)

index = load_index(str(ROOT / "data" / "inverted_index.pkl"))
processed = pd.read_pickle(ROOT / "data" / "processed_corpus.pkl")
titles = processed.set_index("pid")["title"]
print(f"documents: {index.n_docs} | vocabulary: {len(index.vocab)} | rankings: {sorted(RANKINGS_LIST)}")

documents: 28080 | vocabulary: 5053 | rankings: ['bm25', 'tfidf']


## 2. Test queries

The five queries of Part 2, with the number of documents matched by the AND retrieval.

In [3]:
queries = {
    "q1": "men round neck cotton tshirt",
    "q2": "women casual cotton blend topwear",
    "q3": "men slim fit blue jeans",
    "q4": "black solid cotton polo",
    "q5": "women full sleeve printed sweatshirt",
}
pd.DataFrame([{"query": qid, "text": q, "terms": process_query(q),
               "AND matches": len(conjunctive_retrieve(process_query(q), index))}
              for qid, q in queries.items()])

,query,text,terms,AND matches
0,q1,men round neck cotton tshirt,"[men, round, neck, cotton, tshirt]",3678
1,q2,women casual cotton blend topwear,"[women, casual, cotton, blend, topwear]",1548
2,q3,men slim fit blue jeans,"[men, slim, fit, blue, jean]",104
3,q4,black solid cotton polo,"[black, solid, cotton, polo]",222
4,q5,women full sleeve printed sweatshirt,"[women, full, sleev, print, sweatshirt]",341


## 3. Ranking TF-IDF + cosine similarity

Query and documents are vectors over the vocabulary with weights `(1 + log tf) · log(N / df)`. A document's score is the cosine of the angle between its vector and the query vector.

In [15]:
def show(query, k=10, ranking="tfidf"):
    results = search_in_corpus(query, index, top_k=k, ranking=ranking)
    print(f"Query: {query!r} -> terms {process_query(query)} | ranking: {ranking}")
    return pd.DataFrame([(pid, titles[pid], round(score, 4)) for pid, score in results],
                        columns=["pid", "title", "score"])

show(queries["q3"], k=5)

Query: 'men slim fit blue jeans' -> terms ['men', 'slim', 'fit', 'blue', 'jean'] | ranking: tfidf


,pid,title,score
0,JEAF8HYN5B9ZU6FP,Slim Men Blue Jeans,0.4831
1,JEAF2QF6HNZHDXVX,Slim Men Dark Blue Jeans,0.4703
2,JEAEVJGUSXRNSHRY,Slim Men Dark Blue Jeans,0.4474
3,JEAFJA4M7SMZN2XJ,Slim Men Black Jeans,0.4352
4,JEAF8HYNUMGZMQU5,Slim Men Blue Jeans,0.4221


## 3.1 Evaluation

The judgments are the labels of Part 2 (`test_queries_labels.csv`, top 20 of the TF-IDF ranking per query). Only judged documents count, as in Part 2, so the table is comparable across methods.

In [5]:
labels_df = pd.read_csv(ROOT / "project_progress" / "part_2" / "test_queries_labels.csv")
judgments = {qid: dict(zip(g["pid"], g["label"])) for qid, g in labels_df.groupby("query_id")}
KS = (5, 10, 20)

def evaluate(ranking):
    """Per-query and global metrics of one ranking method over the test queries."""
    rows = []
    for qid, q in queries.items():
        ranked = [pid for pid, _ in search_in_corpus(q, index, ranking=ranking)]
        judged = judgments[qid]
        labels = labels_in_rank_order(ranked, judged)
        n_rel = count_relevant(list(judged.values()))
        row = {"ranking": ranking, "query": qid, "AP": average_precision(labels, n_rel), "RR": reciprocal_rank(labels)}
        for k in KS:
            row[f"P@{k}"] = precision_at_k(labels, k)
            row[f"AP@{k}"] = average_precision_at_k(labels, k, n_rel)
            row[f"NDCG@{k}"] = ndcg_at_k(labels, k, list(judged.values()))
        rows.append(row)
    return pd.DataFrame(rows)

per_query = evaluate("tfidf")
per_query.round(3)

,ranking,query,AP,RR,P@5,AP@5,NDCG@5,P@10,AP@10,NDCG@10,P@20,AP@20,NDCG@20
0,tfidf,q1,1.000,1.0,1.0,1.00,1.000,1.0,1.000,1.000,1.00,1.000,1.000
1,tfidf,q2,1.000,1.0,1.0,1.00,1.000,1.0,1.000,1.000,1.00,1.000,1.000
2,tfidf,q3,0.771,1.0,0.8,0.76,0.854,0.7,0.604,0.763,0.55,0.771,0.924
3,tfidf,q4,1.000,1.0,1.0,1.00,1.000,1.0,1.000,1.000,1.00,1.000,1.000
4,tfidf,q5,1.000,1.0,1.0,1.00,1.000,1.0,1.000,1.000,1.00,1.000,1.000


## 4. Ranking: BM25

BM25 scores each query term with a saturating term frequency and a length normalisation, and adds the scores:

`score(q,d) = Σ idf(t) · tf · (k1 + 1) / (tf + k1 · (1 − b + b · |d| / avgdl))`, with `idf(t) = log(1 + (N − df + 0.5) / (df + 0.5))`.

`k1 = 1.5` limits the gain of repeated terms and `b = 0.75` penalises documents longer than the average. Unlike cosine, a long document is not divided by its norm, so BM25 should be less biased towards short documents.

In [14]:
show(queries["q3"], k=5, ranking="bm25")

Query: 'men slim fit blue jeans' -> terms ['men', 'slim', 'fit', 'blue', 'jean'] | ranking: bm25


,pid,title,score
0,JEAF2QF6HNZHDXVX,Slim Men Dark Blue Jeans,13.1076
1,JEAF8HYN5B9ZU6FP,Slim Men Blue Jeans,12.5379
2,JEAF8HYNUMGZMQU5,Slim Men Blue Jeans,12.2384
3,JEAEVJGUSXRNSHRY,Slim Men Dark Blue Jeans,12.1067
4,JEAFS2JZ3RY889ZS,Slim Men Blue Jeans,12.0789


### 4.1 Evaluation

Same labels and metric code as for TF-IDF. Documents outside the TF-IDF top 20 are unjudged and skipped.

In [13]:
per_query = pd.concat([evaluate("tfidf"), evaluate("bm25")], ignore_index=True)
per_query[per_query["query"].isin(["q3"])].round(3)

,ranking,query,AP,RR,P@5,AP@5,NDCG@5,P@10,AP@10,NDCG@10,P@20,AP@20,NDCG@20
2,tfidf,q3,0.771,1.0,0.8,0.76,0.854,0.7,0.604,0.763,0.55,0.771,0.924
7,bm25,q3,0.931,1.0,1.0,1.00,1.000,0.9,0.900,0.936,0.55,0.931,0.982


In [16]:
def summary(per_query_df):
    return pd.DataFrame([{"ranking": r, "MAP": mean_average_precision(g["AP"].tolist()),
                          "MRR": mean_reciprocal_rank(g["RR"].tolist()), "mean NDCG@10": g["NDCG@10"].mean()}
                         for r, g in per_query_df.groupby("ranking", sort=False)]).round(3)

summary(per_query)

,ranking,MAP,MRR,mean NDCG@10
0,tfidf,0.954,1.0,0.953
1,bm25,0.986,1.0,0.987


## 5. Ranking: our own score

BM25 only looks at the text. In a shop, between two products that match the query almost equally well, the one that customers rate better is a better result. Our score starts from BM25 and adds the average rating of the product:

`score(q,d) = 0.9 · bm25(q,d) / max_bm25 + 0.1 · rating(d) / 5`

- `bm25 / max_bm25`: BM25 divided by the best BM25 among the matches of the query, so it is between 0 and 1.
- `rating / 5`: the rating, also between 0 and 1. A product without rating gets the mean rating of the corpus (3.63), so it is neither pushed up nor down.
- The weights 0.9 and 0.1 make the text decide. The rating can change the score by 0.08 at most, so it only reorders documents with almost the same BM25.


In [9]:
from myapp.search.algorithms import load_ratings, RATINGS

load_ratings(processed)
results = show(queries["q1"], k=10, ranking="custom")
results["rating"] = [RATINGS[pid] for pid in results["pid"]]
results


Query: 'men round neck cotton tshirt' -> terms ['men', 'round', 'neck', 'cotton', 'tshirt'] | ranking: custom


,pid,title,score,rating
0,TSHFUNN2H8DUMJYQ,Printed Men Round Neck Blue T-Shirt,0.9740,3.7
1,TSHFUNN2WF5PB3NZ,Printed Men Round Neck White T-Shirt,0.9740,3.7
2,TSHFUNN3QMCGSNCD,Printed Men Round Neck Pink T-Shirt,0.9740,3.7
3,TSHFY6VAJT97F8EA,Printed Men Round Neck Black T-Shirt,0.9636,4.1
4,TSHFMB6CDQ2UB8X3,Printed Men Round Neck Black T-Shirt,0.9616,4.0
5,TSHFMB6CRCHGYZWK,Printed Men Round Neck White T-Shirt,0.9616,4.0
6,TSHFMV7FGH6AZVUN,Printed Men Round Neck Blue T-Shirt,0.9616,4.0
7,TSHFUZCDJWUMMMHF,Printed Men Round Neck Grey T-Shirt,0.9616,4.0
8,TSHFVXGQ8Z5ZHCBS,Printed Men Round Neck Yellow T-Shirt,0.9609,3.2
9,TSHFVXGQG7SSCQFH,Printed Men Round Neck Black T-Shirt,0.9609,3.2


### 5.1 Evaluation

Same labels and metric code as before. The labels only say if the product matches the query, not if it is a good product, so the rating cannot improve these metrics. What we want to check is that the relevance stays close to BM25.


In [10]:
per_query = pd.concat([evaluate("tfidf"), evaluate("bm25"), evaluate("custom")], ignore_index=True)
summary(per_query)


,ranking,MAP,MRR,mean NDCG@10
0,tfidf,0.954,1.0,0.953
1,bm25,0.986,1.0,0.987
2,custom,0.985,1.0,0.987


What the rating changes: the mean rating of the top 10 of each method, per query.


In [11]:
rows = []
for qid, q in queries.items():
    row = {"query": qid}
    for ranking in ["tfidf", "bm25", "custom"]:
        top10 = search_in_corpus(q, index, top_k=10, ranking=ranking)
        total = 0
        for pid, score in top10:
            total += RATINGS[pid]
        row[ranking] = round(total / len(top10), 2)
    rows.append(row)
pd.DataFrame(rows)


,query,tfidf,bm25,custom
0,q1,3.90,3.76,3.76
1,q2,3.67,3.56,4.27
2,q3,3.44,3.56,3.69
3,q4,4.16,3.49,4.11
4,q5,3.55,3.56,3.86


## 6. Ranking: word2vec + cosine similarity

Word2vec gives every term a vector of 100 numbers, learned from the words around it: terms used in similar contexts get similar vectors. We train it on the `terms` of our own corpus, the same processed terms of the index, so the query terms (processed with the same pipeline) are found in the model.

A text is represented by the average of the vectors of its terms. The query and each document matching the AND retrieval get one vector each, and the score is the cosine between them.


In [12]:
from myapp.search.algorithms import train_word2vec

model = train_word2vec(processed)
print(f"terms with a vector: {len(model.wv)} | dimensions: {model.vector_size}")
pd.DataFrame({term: [w for w, _ in model.wv.most_similar(term, topn=5)] for term in ["jean", "tshirt", "sweatshirt"]})


terms with a vector: 5053 | dimensions: 100


,jean,tshirt,sweatshirt
0,denim,neck,fleec
1,chino,polo,hoodi
2,trouser,round,pullov
3,loafer,print,sweater
4,skinni,half,hood


### Top 20 of each test query


In [13]:
for qid, q in queries.items():
    display(show(q, k=20, ranking="word2vec"))


Query: 'men round neck cotton tshirt' -> terms ['men', 'round', 'neck', 'cotton', 'tshirt'] | ranking: word2vec


,pid,title,score
0,TSHFVXGQ8Z5ZHCBS,Printed Men Round Neck Yellow T-Shirt,0.9737
1,TSHFMB6CDQ2UB8X3,Printed Men Round Neck Black T-Shirt,0.9736
2,TSHFY6VAJT97F8EA,Printed Men Round Neck Black T-Shirt,0.9736
3,TSHFUZCDJWUMMMHF,Printed Men Round Neck Grey T-Shirt,0.9736
4,TSHFMB6CRCHGYZWK,Printed Men Round Neck White T-Shirt,0.9735
5,TSHFVXGQG7SSCQFH,Printed Men Round Neck Black T-Shirt,0.9728
6,TSHFMV7FGH6AZVUN,Printed Men Round Neck Blue T-Shirt,0.9727
7,TSHFY6VAUGZFRHDY,Printed Men Round Neck Dark Blue T-Shirt,0.9715
8,TSHFUNN2WF5PB3NZ,Printed Men Round Neck White T-Shirt,0.9707
9,TSHFUNN3QMCGSNCD,Printed Men Round Neck Pink T-Shirt,0.9706


Query: 'women casual cotton blend topwear' -> terms ['women', 'casual', 'cotton', 'blend', 'topwear'] | ranking: word2vec


,pid,title,score
0,SHTFPKNYZUTKXTGR,Women Regular Fit Solid Casual Shirt,0.9224
1,SHTFS6WHVJQFVUHP,Women Slim Fit Printed Casual Shirt,0.9129
2,SHTFAHEYNJFTZDNN,Women Regular Fit Solid Casual Shirt,0.9117
3,SHTFQQHGGY69XHV4,Women Regular Fit Solid Casual Shirt,0.9114
4,SHTFYTWBSSATGV78,Women Slim Fit Solid Casual Shirt,0.9101
5,SHTFS6WCXPRJBMSJ,Women Slim Fit Printed Casual Shirt,0.9096
6,SHTFSYG8GA35ECDT,Women Slim Fit Printed Casual Shirt,0.9096
7,SHTFQQHGA4HUHWYQ,Women Regular Fit Solid Casual Shirt,0.9095
8,SHTFP8J3EGDJS2BW,Women Regular Fit Printed Casual Shirt,0.9088
9,SHTFTJZ4RETDZMQH,Women Regular Fit Solid Casual Shirt,0.9087


Query: 'men slim fit blue jeans' -> terms ['men', 'slim', 'fit', 'blue', 'jean'] | ranking: word2vec


,pid,title,score
0,JEAF8HYN5B9ZU6FP,Slim Men Blue Jeans,0.9507
1,JEAF8HYNUMGZMQU5,Slim Men Blue Jeans,0.9450
2,JEAFJA4M7SMZN2XJ,Slim Men Black Jeans,0.9410
3,JEAFUZXUGWXHUF5F,Jogger Fit Men Blue Jeans,0.9357
4,JEAF2QF6HNZHDXVX,Slim Men Dark Blue Jeans,0.9336
5,JEAFJM2P2FQQJF3C,Slim Men Dark Blue Jeans,0.9330
6,JEAFS2JZF6ZS6X3G,Slim Men Blue Jeans,0.9319
7,JEAFS2JZ3RY889ZS,Slim Men Blue Jeans,0.9317
8,JEAFS2JZQBHZSHAV,Slim Men Blue Jeans,0.9317
9,JEAEVJGUSXRNSHRY,Slim Men Dark Blue Jeans,0.9316


Query: 'black solid cotton polo' -> terms ['black', 'solid', 'cotton', 'polo'] | ranking: word2vec


,pid,title,score
0,TSHFE4JPXV56KKB2,Solid Women Polo Neck Black T-Shirt,0.9290
1,TSHFYDHZNVWDWBGN,Solid Women Polo Neck Black T-Shirt,0.9228
2,TSHFYDHGKTDYZ6HB,"Solid Men Polo Neck Black, Maroon T-Shirt (Pa...",0.9223
3,TSHFYFAS85QRGDX8,"Solid Men Polo Neck Black, Light Blue T-Shirt ...",0.9223
4,TSHFYFAS85VY8HRA,"Solid Men Polo Neck Grey, Black T-Shirt (Pack...",0.9221
5,TSHFYFATFYAY698Z,"Solid Men Polo Neck Dark Blue, Black T-Shirt ...",0.9217
6,TSHFM2W2VRSDGWPU,Solid Women Polo Neck Black T-Shirt,0.9212
7,TSHFM2XPKV4HWNNZ,Solid Women Polo Neck Black T-Shirt,0.9212
8,TSHFP5Y6V7QZG5TJ,Solid Women Polo Neck Black T-Shirt,0.9212
9,TSHF3UHYNWES4KUH,"Solid Men Polo Neck Black, Grey T-Shirt",0.9211


Query: 'women full sleeve printed sweatshirt' -> terms ['women', 'full', 'sleev', 'print', 'sweatshirt'] | ranking: word2vec


,pid,title,score
0,SWSFQGS456JAZCQB,Full Sleeve Printed Women Sweatshirt,0.9748
1,SWSFWJFHA66JXUKR,Full Sleeve Printed Women Sweatshirt,0.9745
2,SWSFZVTNGM4HG8BC,Full Sleeve Printed Women Sweatshirt,0.9738
3,SWSF8922CPKGGHKG,Full Sleeve Printed Women Sweatshirt,0.9723
4,SWSFZAUYZFAGEPKK,Full Sleeve Printed Women Sweatshirt,0.9714
5,SWSFVN9JGHHCGDDS,Full Sleeve Printed Women Sweatshirt,0.9714
6,SWSF8922GAMUHF2H,Full Sleeve Printed Women Sweatshirt,0.9711
7,SWSF89227UCGMH6W,Full Sleeve Printed Women Sweatshirt,0.9709
8,SWSFY8GP8DGV6B5J,Full Sleeve Printed Women Sweatshirt,0.9699
9,SWSEXAHDQQCKCAEA,Full Sleeve Printed Women Sweatshirt,0.9689


### 6.1 Evaluation

Same labels and metric code as for the other rankings (only judged documents count).


In [ ]:
per_query = pd.concat([evaluate("tfidf"), evaluate("bm25"), evaluate("custom"), evaluate("word2vec")], ignore_index=True)
summary(per_query)


,ranking,MAP,MRR,mean NDCG@10
0,tfidf,0.954,1.0,0.953
1,bm25,0.986,1.0,0.987
2,custom,0.985,1.0,0.987
3,word2vec,0.950,1.0,0.959
